# Portfolio — IS risk match

Align S1, S2, and S3 on research IS. Risk-match by inverse volatility. Review correlations. Half-Kelly is a ceiling on the combined IS Sharpe and vol, not a second lever on sealed returns.

CAGR is the path at the stated vol target. Compare sleeves on Sharpe, max drawdown, and Calmar.

The monthly table is a paper report against SPY on the same dates. It does not choose the vol target.


## 0. Imports & Config


In [ ]:
import os
import sys

import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.equity_fetcher import fetch_ohlcv
from performance.sharpe_inference import return_moments
from risk.analytics.leverage.policy import half_kelly_target_vol

CANDIDATES = {
    "s1": os.path.join(ROOT, "01_data", "data_files", "s1_equities", "s1_period_returns.parquet"),
    "s2": os.path.join(ROOT, "01_data", "data_files", "s2_coint", "v1", "s2_period_returns.parquet"),
    "s3": os.path.join(ROOT, "01_data", "data_files", "s3_trend", "s3_period_returns.parquet"),
}


## 1. Load IS return series


In [ ]:
frames = {}
for name, path in CANDIDATES.items():
    if not os.path.isfile(path):
        print("missing", name, path)
        continue
    df = pd.read_parquet(path)
    col = "ret" if "ret" in df.columns else df.columns[0]
    s = pd.to_numeric(df[col], errors="coerce")
    if "date" in df.columns:
        s.index = pd.to_datetime(df["date"])
    else:
        s.index = pd.to_datetime(df.index)
    frames[name] = s.sort_index().dropna()
    print(name, "n", len(frames[name]))
if len(frames) < 2:
    raise FileNotFoundError("Need at least two sleeve return files exported from the IS tearsheets.")
book = pd.DataFrame(frames).dropna(how="any")
print("overlap", book.shape)


## 2. Inverse-vol weights and half-Kelly ceiling


In [ ]:
vol = book.std(ddof=1)
inv = 1.0 / vol.replace(0.0, np.nan)
w = inv / inv.sum()
display(w.rename("inv_vol_weight"))
display(book.corr())
mix = book.mul(w, axis=1).sum(axis=1)
# Weekly S1 and daily S2/S3 should be resampled to one clock before this is live.
ppy = 252.0 if book.index.to_series().diff().dt.days.median() < 3 else 52.0
hk = half_kelly_target_vol(mix, periods_per_year=ppy)
print("half-Kelly vol ceiling", round(hk, 4), "periods/year", ppy)
print("Apply this ceiling on research IS only. Do not refit it on the holdout.")

def _path_stats(s: pd.Series) -> dict:
    r = pd.to_numeric(s, errors="coerce").dropna()
    eq = (1.0 + r).cumprod()
    dd = float((eq / eq.cummax() - 1.0).min()) if len(eq) else float("nan")
    mom = return_moments(r, periods_per_year=ppy)
    ann = float((1.0 + r).prod() ** (ppy / max(len(r), 1)) - 1.0)
    calmar = ann / abs(dd) if dd < 0 else float("nan")
    return {"sharpe": mom["sr"], "max_dd": dd, "calmar": calmar, "cagr": ann}

rows = {name: _path_stats(book[name]) for name in book.columns}
rows["mix"] = _path_stats(mix)
display(pd.DataFrame(rows).T)


## 3. Monthly paper report vs SPY

Month-end compounded returns of the inverse-vol mix, next to SPY over the same overlap. SPY is the benchmark path. It does not set weights or the vol ceiling.


In [ ]:
monthly = (1.0 + mix).resample("ME").prod() - 1.0
report = monthly.rename("mix").to_frame()
start = book.index.min().strftime("%Y-%m-%d")
end = book.index.max().strftime("%Y-%m-%d")
try:
    spy = fetch_ohlcv("SPY", start, end)
    spy = spy.set_index("date")["close"].sort_index().pct_change().dropna()
    spy_m = (1.0 + spy).resample("ME").prod() - 1.0
    report = report.join(spy_m.rename("spy"), how="left")
    report["mix_minus_spy"] = report["mix"] - report["spy"]
except Exception as exc:
    print("SPY benchmark unavailable:", exc)
display(report.tail(12))
print("months", len(report), "mix total", float((1.0 + report["mix"]).prod() - 1.0))
